# Baseline｜三种静态策略正式样本外评估

本 Notebook 只读取已经冻结且通过完整性校验的 OOS evaluation artifact，不重新训练模型、不重新生成 Test scores，也不改变任何策略或统计定义。报告比较等权、固定 ICIR 和 LightGBM 三种静态策略在 2021–2025 年真实样本外区间的表现。

阅读顺序为：样本覆盖 → 评分有效性 → 分组单调性 → 多头与超额收益 → 换手率 → LightGBM 跨阶段稳定性 → 冻结信息。

## 00｜参数与权威性校验

这里绑定本次正式 Baseline 的 OOS evaluation manifest，以及完整 Factor Pool、Static Strategy Bundle 和 Test Score Artifact 的权威指纹。任一指纹不一致都会拒绝加载。输出目录按 evaluation fingerprint 隔离，避免覆盖其他实验。

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from factor_gfn.backtest import load_verified_oos_baseline_evaluation
from factor_gfn.reporting import build_oos_report_data, OOSReportRenderer

EVALUATION_MANIFEST = Path(
    r'runs/oos_baseline_evaluations/'
    r'9d271368528d002a8af0807c807042a0e93f6e6b21e5190d65a378deacdc7951/'
    r'evaluation_manifest.json'
)
FACTOR_POOL_FINGERPRINT = (
    'f9a3945945ee04eb357896b7b8e20d63db4a8a9a8db5c3a2a10820a70ab211d4'
)
STRATEGY_BUNDLE_FINGERPRINT = (
    '5e058b0ad182ef329584ee060d22d3f8d2d070c561dcf3e86168c100804263d3'
)
TEST_SCORE_ARTIFACT_FINGERPRINT = (
    '63d77fbd3bf23aaccbd8a25c38cc27b79ddc895f1d5514679ddc63b2586433c0'
)
OUTPUT_ROOT = Path(r'outputs/oos_baseline')

evaluation = load_verified_oos_baseline_evaluation(
    EVALUATION_MANIFEST,
    expected_factor_pool_fingerprint=FACTOR_POOL_FINGERPRINT,
    expected_strategy_bundle_fingerprint=STRATEGY_BUNDLE_FINGERPRINT,
    expected_test_score_artifact_fingerprint=TEST_SCORE_ARTIFACT_FINGERPRINT,
)
report = build_oos_report_data(evaluation)
renderer = OOSReportRenderer(report, OUTPUT_ROOT / evaluation.fingerprint)

中文列名 = {
    'Strategy': '策略', 'Sample': '样本', 'Split': '阶段',
    'Mean RankIC': '平均 RankIC', 'ICIR': 'ICIR',
    'G10 Geometric Annualized Return': 'G10 几何年化收益率',
    'G10 Annualized Return': 'G10 年化收益率',
    'G10 Annualized Volatility': 'G10 年化波动率',
    'G10 Sharpe': 'G10 夏普比率', 'G10 Max Drawdown': 'G10 最大回撤',
    'Excess Geometric Annualized Return': '超额几何年化收益率',
    'Excess Annualized Return': '超额年化收益率',
    'Excess Annualized Volatility': '超额年化波动率',
    'Excess Sharpe': '超额夏普比率', 'Excess Max Drawdown': '超额最大回撤',
    'Excess Win Rate': '超额胜率',
    'Mean One-Way Turnover': '平均单边换手率',
    'Median': '中位数', 'Max': '最大值', 'Turnover Observations': '换手观测数',
    'Mean Constituent Replacement Rate': '平均成分替换率',
    'Mean Raw Universe Count': '平均原始股票数',
    'Mean Complete-Case Eligible Count': '平均基础合格股票数',
    'Mean Label-Eligible Count': '平均标签有效股票数',
    'Min Eligible Count': '最少基础合格股票数',
    'Mean Coverage Ratio': '平均覆盖率', 'Min Coverage Ratio': '最低覆盖率',
    'Invalid Rebalance Periods': '无效调仓期数',
    'Field': '字段', 'Value': '取值',
}
中文取值 = {
    'Equal Weight': '等权', 'Fixed ICIR': '固定 ICIR',
    'Train': '训练集', 'Validation': '验证集', 'Test': '测试集',
    'common_sample': '统一样本',
}

def 显示中文表格(dataframe):
    return dataframe.rename(columns=中文列名).replace(中文取值)

print('OOS 评估状态：', evaluation.manifest['evaluation_status'])
print('OOS 评估指纹：', evaluation.fingerprint)
print('样本外日期：', evaluation.manifest['key_ranges']['coverage_by_date'])
print('报告输出目录：', renderer.output_dir)


> **当前策略矩阵口径**：Raw expression → 1%/99% 截面缩尾 → PIT 申万一级行业中性化 → 截面标准化 → base-eligible 股票内的因子特定缺失值填 0 → 冻结 Train direction → Top100 Strategy Matrix。
>
> 当前 evaluation manifest 和底层 reporting 数据中仍保留旧字段名 `complete_case` / `Mean Complete-Case Eligible Count`，这是兼容性命名残留；本 Notebook 将其显示为“基础合格股票”，实际结果并不是重新要求 Top100 全部同时非缺失。

## 01｜样本覆盖

该图回答策略在每个调仓日实际覆盖了多少只股票。上图比较原始股票池与满足基础资格条件的股票数量，下图展示覆盖率。重点检查是否存在覆盖率突然接近 0、股票数异常断层或大量无效调仓期。这里的基础资格主要由正式股票池和可用的 PIT 行业标签决定，因子特定缺失已经在 cleaning 后填 0。

In [ ]:
renderer.figure_coverage()

In [ ]:
显示中文表格(report.coverage_summary)

## 02｜策略评分有效性

周期 RankIC 衡量每个调仓截面中策略分数与未来收益排序的一致性；累计 IC 只是 RankIC 的累计和，不是净值。策略分数相关性热力图用于判断三种策略是否实际上给出了高度相似的股票排序。相关性高代表信号冗余较强，但不能单独判断哪种策略更有效。热力图保留三位小数，避免把接近 1 的相关性误读为严格等于 1；下方同时显示四位小数的精确矩阵。

In [ ]:
renderer.figure_rank_ic()

In [ ]:
renderer.figure_score_correlation()

In [ ]:
相关矩阵中文 = report.strategy_score_correlation.rename(
    index={'Equal Weight': '等权', 'Fixed ICIR': '固定 ICIR'},
    columns={'Equal Weight': '等权', 'Fixed ICIR': '固定 ICIR'},
)
display(相关矩阵中文.style.format('{:.4f}'))

## 03｜十分位组合分析

每个调仓日按策略分数从低到高分为 G1–G10，G10 代表策略最看好的股票。主图展示各组未来 5 日收益减去同日 evaluation-eligible 股票池等权基准后的平均超额收益；三个策略使用完全相同的基准，0 轴用于判断各分组相对同日市场截面的超额方向。判断横截面区分能力时，应联合观察分组单调性与 G10−G1。原始绝对收益仍保留在冻结 OOS artifact 中，本图仅改变展示口径。

In [ ]:
renderer.figure_decile_return()

In [ ]:
显示中文表格(report.decile_return_table)

In [ ]:
策略中文名 = {'equal_weight': '等权', 'fixed_icir': '固定 ICIR', 'lightgbm': 'LightGBM'}
分组列 = [f'G{i}' for i in range(1, 11)]
平均分组绝对收益 = report.decile_returns.groupby('strategy_id')[分组列].mean()
平均同期基准收益 = report.portfolio_returns.groupby('strategy_id')['benchmark_return'].mean()
平均分组超额收益 = 平均分组绝对收益.sub(平均同期基准收益, axis=0).rename(index=策略中文名)
print('各组相对同日全样本等权基准的平均 5 日超额收益：')
display(平均分组超额收益.style.format('{:.2%}').background_gradient(cmap='RdYlGn', axis=None))

## 04｜G10 多头组合与基准

这里比较三种策略各自 G10 多头组合的累计净值，并加入同一合格股票样本上的等权基准。该图展示的是未扣交易成本的毛收益净值，因此需要结合后续换手率判断策略在实际交易中的可实现性。

In [ ]:
renderer.figure_g10_nav()

## 05｜超额收益与多空收益

超额净值定义为 G10 相对统一等权基准的累计表现；多空净值定义为 G10−G1。前者更接近多头选股价值，后者更集中反映评分横截面区分能力。主绩效表汇总 RankIC、ICIR、年化收益、波动率、最大回撤、胜率和换手率。

In [ ]:
renderer.figure_excess_nav()

In [ ]:
renderer.figure_long_short_nav()

In [ ]:
显示中文表格(report.main_strategy_performance_summary)

### 三种冻结策略样本外绩效对比

该补充图仿照既有 LightGBM 因子组合报告，把三种策略在完全相同 Test 样本上的核心指标放入同一色阶表。每一列独立按三种策略的相对表现着色；除换手率越低越好外，其余列均按数值越高越好处理。最大回撤为负数，因此更接近 0 的值颜色更优。所有收益均为未扣交易成本的毛收益。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

绩效列 = [
    ('Mean RankIC', '平均 RankIC', False),
    ('ICIR', 'ICIR', False),
    ('Geometric Annualized Excess Return', '年化超额', True),
    ('Excess IR', '超额 IR', False),
    ('G10 Max Drawdown', '最大回撤', True),
    ('Excess Win Rate', '超额胜率', True),
    ('G10-G1 Geometric Annualized Return', '年化多空', True),
    ('Mean One-Way Turnover', '换手率', True),
]
绩效数据 = report.main_strategy_performance_summary.set_index('Strategy')
绩效数据 = 绩效数据.rename(index={'Equal Weight': '等权', 'Fixed ICIR': '固定 ICIR'})
策略顺序 = ['LightGBM', '固定 ICIR', '等权']
绩效数据 = 绩效数据.loc[策略顺序, [item[0] for item in 绩效列]]

def 格式化绩效(value, percentage):
    return f'{value:.2%}' if percentage else f'{value:.4f}'

中文字体路径 = font_manager.findfont('Microsoft YaHei', fallback_to_default=False)
中文字体 = font_manager.FontProperties(fname=中文字体路径)
with plt.rc_context({'axes.unicode_minus': False}):
    figure, axis = plt.subplots(figsize=(14.5, 4.6))
    axis.axis('off')
    axis.set_title('三种冻结策略样本外绩效对比', fontsize=16, pad=18, fontproperties=中文字体)
    单元格文字 = [
        [格式化绩效(row[column], percentage) for column, _, percentage in 绩效列]
        for _, row in 绩效数据.iterrows()
    ]
    table = axis.table(
        cellText=单元格文字, rowLabels=绩效数据.index,
        colLabels=[label for _, label, _ in 绩效列],
        cellLoc='center', rowLoc='center', bbox=[0.0, 0.09, 1.0, 0.78],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    色图 = plt.get_cmap('RdYlGn')
    for (row_index, column_index), cell in table.get_celld().items():
        cell.set_edgecolor('white')
        cell.get_text().set_fontproperties(中文字体)
        if row_index == 0:
            cell.set_facecolor('#294E68')
            cell.get_text().set_color('white')
            cell.get_text().set_weight('bold')
        elif column_index == -1:
            cell.set_facecolor('#E6EDF2')
            cell.get_text().set_weight('bold')
        else:
            column = 绩效列[column_index][0]
            values = 绩效数据[column].to_numpy(dtype=float)
            lower, upper = np.nanmin(values), np.nanmax(values)
            scaled = 0.5 if np.isclose(lower, upper) else (绩效数据.iloc[row_index - 1][column] - lower) / (upper - lower)
            if column == 'Mean One-Way Turnover':
                scaled = 1.0 - scaled
            base = 色图(float(np.clip(scaled, 0.0, 1.0)))
            cell.set_facecolor(tuple(0.42 * component + 0.58 for component in base[:3]))
    axis.text(
        0.5, 0.01,
        '三种策略使用完全相同的 2021–2025 Test 股票—调仓日样本；G10 为最高得分组；结果未扣交易成本。',
        ha='center', va='bottom', fontsize=9, color='#555555', transform=axis.transAxes, fontproperties=中文字体,
    )
    figure.tight_layout()

补充图目录 = renderer.output_dir / 'supplementary'
补充图目录.mkdir(parents=True, exist_ok=True)
绩效对比图路径 = 补充图目录 / '01_oos_strategy_performance_comparison.png'
figure.savefig(绩效对比图路径, dpi=170, bbox_inches='tight')
print('补充绩效对比图：', 绩效对比图路径)
figure

## 06｜换手率

换手率采用持仓漂移调整后的单边口径。平均换手率用于比较三种策略的总体交易强度，时间序列用于识别阶段性跳升、组合重置或不稳定时期。首期以及重置期可能为 NaN，这是口径设计而不是计算错误。

In [ ]:
renderer.figure_average_turnover()

In [ ]:
renderer.figure_turnover_series()

In [ ]:
显示中文表格(report.turnover_summary)

## 07｜LightGBM 跨阶段诊断

该图并列展示 LightGBM 在训练集、验证集和真实测试集上的平均 RankIC 与 ICIR。验证集承担 Development 阶段早停的角色，因此不是最终 OOS；测试集才是模型冻结后首次访问的真实 OOS。重点观察方向是否一致以及从 Development 到 Test 的衰减程度。

In [ ]:
renderer.figure_lightgbm_splits()

In [ ]:
显示中文表格(report.lightgbm_split_effectiveness)

## 08｜策略冻结与可复现信息

本表记录 Factor Pool、Strategy Bundle、Test Score Artifact、固定 ICIR 权重及 LightGBM 模型的关键身份信息。它用于证明三种策略在首次读取 Test labels 之前已经冻结，不用于比较策略绩效。

In [ ]:
显示中文表格(report.strategy_freeze_summary)

## 09｜导出正式报告

最后统一生成全部图、CSV 表格和 report manifest。前面的单图 Cell 仅用于交互查看；本 Cell 才是完整报告导出入口。导出不会重新训练策略或重新计算 Test scores。

In [ ]:
report_manifest_path = renderer.render_all()
print('正式 OOS 报告 manifest：', report_manifest_path)
print('图表目录：', report_manifest_path.parent / 'figures')
print('表格目录：', report_manifest_path.parent / 'tables')
report_manifest_path